# Continuous HI Cosmology Bias Probe Check

This notebook checks whether the continuous-cosmology HI diffusion model actually follows the requested input cosmology.

The default calibration in this notebook now uses the best current probe:

```text
normalized HI slice -> frozen VGG16 features -> avg+max pooling -> MLP(1024,512,256) -> recovered cosmology
```

The main poster-level result is the `Omega_m` calibration: generated HI fields from the large-data model recover the requested `Omega_m` much better than the small-data model.

The notebook auto-detects the repo root, so it works whether Jupyter starts in `/home/jiamingp/diffusion_models_repo` or `/home/jiamingp/diffusion_models_repo/notebooks`. You can also override detection with `PROJECT_DIR=/home/jiamingp/diffusion_models_repo`.

## What The Experiment Tests

The question is whether a continuous-cosmology-conditioned diffusion model actually uses the input cosmology, or whether it drifts back toward the training distribution.

- Field: HI only. Mstar and Mtot are not part of this continuous-cosmology run.
- Conditioning: the six CAMELS parameters, in file order: $\Omega_\mathrm{m}$, $\sigma_8$, $A_{\mathrm{SN1}}$, $A_{\mathrm{AGN1}}$, $A_{\mathrm{SN2}}$, $A_{\mathrm{AGN2}}$.
- Architecture: `UNet2DConditionModel` with cross-attention. The normalized parameter vector has shape `(B, 6)` and is passed to the model as `encoder_hidden_states` with shape `(B, 1, 6)`.
- Two regimes: `N=128` 2D training fields for the memorization-regime model, and `N=16384` 2D training fields for the generalization-regime model.
- Held-out cosmologies: fixed simulation indices are excluded from both diffusion training and encoder training. Bias is measured only on generated samples conditioned on these held-out cosmologies.
- Main v1 has CFG off: `cfg_dropout=0.0` during training and `guidance_scale=None` during sampling.

## Training Objective

Each training example is a normalized 2D HI slice `x_0` and a normalized cosmology vector `theta`.

At each step, the scheduler samples a diffusion timestep `t` and Gaussian noise `epsilon`, creates a noisy image `x_t`, and asks the conditional UNet to predict the scheduler target. For these configs the scheduler uses `prediction_type: v_prediction`, squared-cosine betas, and Min-SNR weighting with `min_snr_gamma=5.0`.

Conceptually, the loss is a weighted MSE:

`loss = weight(t) * || UNet(x_t, t, theta) - target_v(x_0, epsilon, t) ||^2`.

The important part for this bias test is that the only information about cosmology is the continuous vector `theta` passed through cross-attention. If the model learns the conditioning, generated fields at a held-out `theta` should encode back to that same `theta`. If not, recovered parameters regress toward the training distribution, and the calibration slope is closer to 0 than 1.

## Best VGG Encoder Used Here

The encoder is a diagnostic probe, not part of diffusion training.

Pipeline:

```text
normalized HI field
-> repeat single channel into RGB
-> bilinear resize to 224 x 224
-> frozen ImageNet VGG16 convolutional features
-> average + max pooling
-> MLP regression head: 1024, 512, 256 hidden units
-> six CAMELS parameters
```

Rules enforced by the scripts:

- VGG16 is frozen; its weights are not updated.
- Only the regression head is trained.
- The head is trained on real HI fields only.
- Held-out simulations `900-931` are excluded from encoder fitting and diffusion training.
- Generated fields are never used to fit the encoder.

This is stronger than the older PCA+Ridge probe for the main cosmological parameters. In the completed real-heldout sanity check, the best VGG probe reached roughly `R^2 = 0.91` for `Omega_m` and `R^2 = 0.74` for `sigma_8`.

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_dir() -> Path:
    if os.environ.get('PROJECT_DIR'):
        return Path(os.environ['PROJECT_DIR']).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'scripts').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
        if (candidate / 'results' / 'nf_conditional_bias_probe').exists():
            return candidate
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd


PROJECT_DIR = find_project_dir()
for import_path in [PROJECT_DIR, PROJECT_DIR / 'scripts']:
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

SWEEP_NAME = 'nf_conditional_bias_probe'
RESULT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME
ENC_DIR = RESULT_DIR / 'encoder'

# Default to the best-VGG calibration output. Override if needed with:
# BIAS_PROBE_CALIBRATION_DIR=/path/to/calibration_dir
cal_dir_env = os.environ.get('BIAS_PROBE_CALIBRATION_DIR', '').strip()
CAL_DIR = Path(cal_dir_env).expanduser().resolve() if cal_dir_env else RESULT_DIR / 'calibration_vgg'

POINTS_PATH = CAL_DIR / 'bias_probe_per_cosmology_points.csv'
SAMPLES_PATH = CAL_DIR / 'bias_probe_per_sample_predictions.csv'
SLOPES_PATH = CAL_DIR / 'bias_probe_regime_slopes.csv'
META_PATH = CAL_DIR / 'bias_probe_eval_metadata.json'

BEST_VGG_ENCODER_PATH = ENC_DIR / 'vgg_mlp_big_avgmax.npz'
BEST_VGG_MODEL_PATH = ENC_DIR / 'vgg_mlp_big_avgmax.pkl'
VGG_METRICS_PATH = ENC_DIR / 'vgg_real_test_metrics.csv'
VGG_PRED_PATH = ENC_DIR / 'vgg_real_test_per_cosmology_predictions.csv'
VGG_META_PATH = ENC_DIR / 'vgg_real_test_metadata.json'
VGG_COMPARISON_PATH = ENC_DIR / 'vgg_encoder_r2_comparison.csv'

# Older PCA/Ridge paths are kept only for optional diagnostic sections later in the notebook.
PCA_ENCODER_METRICS_PATH = ENC_DIR / 'encoder_val_metrics.csv'
PCA_ENCODER_SPLIT_PATH = ENC_DIR / 'encoder_real_split.json'

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 260,
    'font.family': 'DejaVu Serif',
    'mathtext.fontset': 'dejavuserif',
    'font.size': 14,
    'axes.titlesize': 17,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 23,
})

print('PROJECT_DIR =', PROJECT_DIR)
print('CAL_DIR     =', CAL_DIR)
for label, path in [
    ('points', POINTS_PATH),
    ('samples', SAMPLES_PATH),
    ('slopes', SLOPES_PATH),
    ('metadata', META_PATH),
    ('best VGG encoder', BEST_VGG_ENCODER_PATH),
    ('VGG real-test metrics', VGG_METRICS_PATH),
]:
    print(f"{label:22s}", 'found' if path.exists() else 'missing', path)

In [ ]:
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


points = read_csv_required(POINTS_PATH)
samples = read_csv_required(SAMPLES_PATH)
slopes = read_csv_required(SLOPES_PATH)
metadata = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}

PARAM_DISPLAY_PRETTY = {
    'Omega_m': 'Ωm',
    'sigma_8': 'σ8',
    'A_SN1': 'A_SN1',
    'A_AGN1': 'A_AGN1',
    'A_SN2': 'A_SN2',
    'A_AGN2': 'A_AGN2',
}


def with_parameter_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'parameter' in out.columns:
        out.insert(out.columns.get_loc('parameter') + 1, 'label', out['parameter'].map(PARAM_DISPLAY_PRETTY).fillna(out['parameter']))
    return out


print('point rows:', len(points), 'sample prediction rows:', len(samples), 'slope rows:', len(slopes))
print('calibration dir:', CAL_DIR)
print('encoder type from metadata:', metadata.get('encoder_type', 'unknown'))
print('vgg encoder from metadata:', metadata.get('vgg_encoder', metadata.get('encoder_path', 'unknown')))
if BEST_VGG_ENCODER_PATH.exists():
    with np.load(BEST_VGG_ENCODER_PATH, allow_pickle=True) as data:
        print('best VGG encoder file:', BEST_VGG_ENCODER_PATH)
        for key in ['encoder_type', 'vgg_weights', 'vgg_pool', 'feature_dim']:
            if key in data:
                value = data[key]
                try:
                    value = value.item()
                except Exception:
                    pass
                print(f'{key}:', value)
        if 'heldout_indices' in data:
            heldout = data['heldout_indices'].astype(int)
            print('heldout sims:', f'{heldout[0]}-{heldout[-1]}', f'(n={len(heldout)})')

sample_count = points['n_samples'].mode().iloc[0] if 'n_samples' in points.columns and len(points) else 'unknown'
print('generated samples per held-out cosmology K =', sample_count)

display(with_parameter_labels(points.head()))
display(with_parameter_labels(slopes.sort_values(['parameter', 'dataset_size'])).round(4))

## Training Length Used Here

The original continuous-conditioning runs were first-pass `200k` update checks. The continued runs extend training, and the current VGG calibration should be read together with one-point PDF, `P(k)`, nearest-training-slice similarity, and training loss.

The key diagnostic question is:

- If simple field statistics are bad, the diffusion model is probably undertrained or low-quality.
- If field statistics are good but recovered-vs-requested slopes are weak, then the model may generate realistic HI while not using the cosmology conditioning strongly enough.
- The VGG probe is stronger than the earlier PCA/Ridge probe, especially for `Omega_m` and `sigma_8`, so it is the better default for the poster calibration story.

In [ ]:
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
manifest_rows = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else []

MAIN_JOB_INFO = {
    'nf_cond_bias_hi_u128_d2p07_n128_200k': {'slurm_job': 51322618, 'elapsed': '11:42:57'},
    'nf_cond_bias_hi_u128_d2p14_n16384_200k': {'slurm_job': 51322619, 'elapsed': '11:42:59'},
}

train_rows = []
for row in manifest_rows:
    if row.get('run_name') not in MAIN_JOB_INFO:
        continue
    info = MAIN_JOB_INFO[row['run_name']]
    train_rows.append({
        'regime': row.get('regime'),
        'run_name': row.get('run_name'),
        'slurm_job': info['slurm_job'],
        'elapsed': info['elapsed'],
        'dataset_size_2d_fields': row.get('dataset_size'),
        'batch_size': row.get('batch_size'),
        'steps_per_epoch': row.get('steps_per_epoch'),
        'epochs': row.get('epochs'),
        'actual_updates': row.get('actual_updates'),
        'checkpoint_every_updates': row.get('checkpoint_every_updates'),
        'cfg_dropout': row.get('cfg_dropout'),
        'conditioning': row.get('conditioning'),
    })
training_summary = pd.DataFrame(train_rows)
display(training_summary)

if len(training_summary):
    out = CAL_DIR / 'bias_probe_training_summary.csv'
    training_summary.to_csv(out, index=False)
    print('wrote', out)
else:
    print('Manifest missing or main runs not found:', MANIFEST_PATH)


## Training Loss Curves

Use this before deciding whether to continue training. Loss is only an optimization-health check; it does not prove good samples by itself. Read it together with one-point PDF, `P(k)`, and nearest-training cosine.

This cell tries two sources: saved `metrics*.json` files under the checkpoint directory, and Slurm logs under `logs/nf_conditional_bias_probe`. The x-axis is optimizer update, not epoch, so `N=128` and `N=16,384` are comparable despite having very different steps per epoch.


In [ ]:
import re


def resolve_project_path(value: str | Path) -> Path:
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_DIR / path


if 'manifest_rows' not in globals():
    MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
    manifest_rows = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else []

if 'MAIN_JOB_INFO' not in globals():
    MAIN_JOB_INFO = {
        'nf_cond_bias_hi_u128_d2p07_n128_200k': {'slurm_job': 51322618, 'elapsed': '11:42:57'},
        'nf_cond_bias_hi_u128_d2p14_n16384_200k': {'slurm_job': 51322619, 'elapsed': '11:42:59'},
    }

if 'main_rows' not in globals():
    main_rows = [row for row in manifest_rows if row.get('run_name') in MAIN_JOB_INFO]

if not main_rows:
    raise RuntimeError('No main bias-probe rows found in manifest. Run the prepare step or check local/nf_conditional_bias_probe/manifest.json.')

if 'REGIME_LABELS' not in globals():
    REGIME_LABELS = {'memorization': 'memorization (N=128)', 'generalization': 'generalization (N=16,384)'}
if 'REGIME_COLORS' not in globals():
    REGIME_COLORS = {'memorization': '#d62728', 'generalization': '#1f77b4'}


def _flatten_numeric(values):
    if values is None:
        return np.asarray([], dtype=float)
    out = []

    def visit(x):
        if x is None:
            return
        if isinstance(x, dict):
            for key in ('loss', 'value', 'mean', 'avg'):
                if key in x:
                    visit(x[key])
                    return
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for item in x:
                visit(item)
            return
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return

    visit(values)
    return np.asarray(out, dtype=float)


def metric_candidates_for_bias(row: dict) -> list[Path]:
    root = resolve_project_path(row.get('checkpoint_dir', ''))
    paths = []
    if root.exists():
        paths.extend(sorted(root.glob('metrics_epoch_*.json')))
        if (root / 'metrics.json').exists():
            paths.append(root / 'metrics.json')
        for ckpt in sorted(root.glob('checkpoint-epoch-*')):
            paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def read_latest_bias_metrics(row: dict) -> tuple[dict, Path | None]:
    paths = metric_candidates_for_bias(row)
    if not paths:
        return {}, None

    def score(path: Path) -> tuple[int, float]:
        m = re.search(r'metrics_epoch_(\d+)', path.name)
        epoch = int(m.group(1)) if m else -1
        try:
            mtime = path.stat().st_mtime
        except OSError:
            mtime = 0.0
        return epoch, mtime

    latest = max(paths, key=score)
    try:
        with latest.open() as f:
            return json.load(f), latest
    except Exception as exc:
        print('could not read metrics JSON:', latest, exc)
        return {}, latest


def log_candidates_for_bias(row: dict) -> list[Path]:
    logs_dir = PROJECT_DIR / 'logs' / SWEEP_NAME
    if not logs_dir.exists():
        return []
    job_id = str(MAIN_JOB_INFO.get(row['run_name'], {}).get('slurm_job', '')).strip()
    patterns = []
    if job_id:
        patterns.extend([f'*{job_id}.out', f'*{job_id}.err'])
    n = int(row['dataset_size'])
    patterns.extend([f'train_n{n}_*.out', f'train_n{n}_*.err'])
    out = []
    seen = set()
    for pattern in patterns:
        for path in sorted(logs_dir.glob(pattern)):
            if path not in seen:
                out.append(path)
                seen.add(path)
    return out


def parse_epoch_losses_from_logs(row: dict) -> tuple[pd.DataFrame, list[Path]]:
    paths = log_candidates_for_bias(row)
    avg_re = re.compile(r'Epoch\s+(\d+)\s+[—\-]\s+avg loss:\s+([0-9.eE+\-]+|nan|inf)', re.IGNORECASE)
    tqdm_re = re.compile(r'Epoch\s+(\d+)/(\d+):.*?loss=([0-9.eE+\-]+)', re.IGNORECASE)
    by_epoch = {}
    for path in paths:
        try:
            text = path.read_text(errors='ignore')
        except OSError:
            continue
        for m in avg_re.finditer(text):
            by_epoch[int(m.group(1))] = float(m.group(2))
        # tqdm logs can contain many updates per epoch; keep the last one seen per epoch.
        for m in tqdm_re.finditer(text.replace('\r', '\n')):
            by_epoch[int(m.group(1))] = float(m.group(3))
    rows = [{'epoch': e, 'avg_loss': v} for e, v in sorted(by_epoch.items()) if np.isfinite(v)]
    return pd.DataFrame(rows), paths


def moving_average(values: np.ndarray, window: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or window <= 1:
        return values
    window = min(window, len(values))
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(values, kernel, mode='valid')


def downsample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 1400) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) <= max_points:
        return x, y
    idx = np.linspace(0, len(x) - 1, max_points, dtype=int)
    return x[idx], y[idx]


loss_rows = []
loss_series = []
for row in main_rows:
    run_name = row['run_name']
    steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1)))
    metrics, metrics_path = read_latest_bias_metrics(row)
    epoch_loss = _flatten_numeric(metrics.get('epoch_loss'))
    batch_loss = _flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
    log_df, log_paths = parse_epoch_losses_from_logs(row)

    source = 'metrics_json' if len(epoch_loss) or len(batch_loss) else ('slurm_log' if len(log_df) else 'missing')
    final_loss = np.nan
    n_points = 0
    if len(epoch_loss):
        final_loss = float(epoch_loss[-1])
        n_points = len(epoch_loss)
        x = np.arange(len(epoch_loss), dtype=float) * steps_per_epoch
        loss_series.append((row, 'epoch loss', x, epoch_loss, source))
    elif len(log_df):
        final_loss = float(log_df['avg_loss'].iloc[-1])
        n_points = len(log_df)
        x = log_df['epoch'].to_numpy(dtype=float) * steps_per_epoch
        y = log_df['avg_loss'].to_numpy(dtype=float)
        loss_series.append((row, 'epoch/log loss', x, y, source))

    if len(batch_loss):
        window = max(1, len(batch_loss) // 1200)
        y = moving_average(batch_loss, window)
        x = np.arange(len(y), dtype=float) + 0.5 * max(0, window - 1)
        loss_series.append((row, 'batch loss', x, y, 'metrics_json'))

    loss_rows.append({
        'regime': row.get('regime'),
        'run_name': run_name,
        'dataset_size': int(row['dataset_size']),
        'slurm_job': MAIN_JOB_INFO.get(run_name, {}).get('slurm_job'),
        'steps_per_epoch': steps_per_epoch,
        'epochs_configured': int(row.get('epochs', 0)),
        'actual_updates_configured': int(row.get('actual_updates', 0)),
        'loss_source': source,
        'n_loss_points': int(n_points),
        'final_loss': final_loss,
        'metrics_path': str(metrics_path) if metrics_path else None,
        'log_paths': '; '.join(str(p) for p in log_paths),
    })

loss_df = pd.DataFrame(loss_rows).sort_values('dataset_size')
display(loss_df.round(6))
loss_summary_out = CAL_DIR / 'bias_probe_training_loss_summary.csv'
loss_df.to_csv(loss_summary_out, index=False)
print('wrote', loss_summary_out)

if not loss_series:
    print('No loss data found. On Great Lakes, check logs/nf_conditional_bias_probe/train_n128_*.out and train_n16384_*.out, or checkpoint metrics JSON.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2), constrained_layout=True)
    for row, kind, x, y, source in loss_series:
        if kind == 'batch loss':
            # Keep the main figure focused on epoch-average loss when available.
            continue
        x_plot, y_plot = downsample_xy(x, y)
        regime = row.get('regime')
        label = f"{REGIME_LABELS.get(regime, regime)}"
        color = REGIME_COLORS.get(regime, 'black')
        axes[0].plot(x_plot, y_plot, lw=2.5, color=color, label=label)
        axes[1].semilogy(x_plot, y_plot, lw=2.5, color=color, label=label)
        if len(y_plot):
            axes[0].annotate(f'{y_plot[-1]:.4g}', (x_plot[-1], y_plot[-1]), textcoords='offset points', xytext=(5, 4), fontsize=11, color=color)
            axes[1].annotate(f'{y_plot[-1]:.4g}', (x_plot[-1], y_plot[-1]), textcoords='offset points', xytext=(5, 4), fontsize=11, color=color)
    axes[0].set_title('Training loss')
    axes[1].set_title('Training loss, log scale')
    for ax in axes:
        ax.set_xlabel('Optimizer update')
        ax.set_ylabel('Mean training loss')
        ax.grid(alpha=0.2)
        ax.legend(frameon=False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.suptitle('Continuous HI bias-probe training loss', y=1.04)
    loss_plot_out = CAL_DIR / 'bias_probe_training_loss_curves.png'
    fig.savefig(loss_plot_out, bbox_inches='tight')
    print('wrote', loss_plot_out)
    plt.show()


In [ ]:
if VGG_METRICS_PATH.exists():
    enc = pd.read_csv(VGG_METRICS_PATH)
    print('VGG encoder real-heldout metrics. Ωm and σ8 are the most important sanity checks; feedback parameters are expected to be weaker.')
    cols = ['split', 'grain', 'parameter', 'n', 'mae', 'rmse', 'bias', 'r2']
    display(with_parameter_labels(enc[cols].sort_values(['split', 'grain', 'parameter'])).round(4))
else:
    print('No VGG encoder validation table found:', VGG_METRICS_PATH)

if VGG_META_PATH.exists():
    meta = json.loads(VGG_META_PATH.read_text())
    print('\nVGG encoder metadata summary:')
    for key in ['vgg_weights', 'vgg_image_size', 'vgg_pool', 'feature_dim', 'head_type', 'head_train_slices', 'head_val_slices', 'test_slices']:
        if key in meta:
            print(f'{key}:', meta[key])
    heldout = meta.get('heldout_indices')
    if heldout:
        print('heldout_indices:', heldout[:4], '...', heldout[-4:], f'(n={len(heldout)})')
else:
    print('No VGG metadata table found:', VGG_META_PATH)

if VGG_COMPARISON_PATH.exists():
    print('\nVGG encoder comparison table:')
    display(pd.read_csv(VGG_COMPARISON_PATH).round(4))

## Cleaner Calibration Plot

Each marker is one held-out input cosmology. The x-axis is the true input parameter. The y-axis is the median recovered parameter after generating many HI samples at that same input cosmology and encoding each generated field back to parameters.

The dashed line is perfect calibration. The fitted line is the main summary: slope near 1 means the generated fields track the input parameter; slope near 0 means the generated fields are nearly independent of the input and regress toward a typical training value.

In [ ]:
PARAM_ORDER = ['Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2']
PARAM_LABELS = {
    'Omega_m': r'$\Omega_m$',
    'sigma_8': r'$\sigma_8$',
    'A_SN1': r'$A_{SN1}$',
    'A_AGN1': r'$A_{AGN1}$',
    'A_SN2': r'$A_{SN2}$',
    'A_AGN2': r'$A_{AGN2}$',
}
REGIME_LABELS = {'memorization': 'Memorization (N=128)', 'generalization': 'Generalization (N=16,384)'}
REGIME_COLORS = {'memorization': '#D55E00', 'generalization': '#0072B2'}
REGIME_MARKERS = {'memorization': 'o', 'generalization': 's'}


def filtered_no_guidance(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'guidance_label' in out.columns and 'noguidance' in set(out['guidance_label'].astype(str)):
        out = out[out['guidance_label'].astype(str) == 'noguidance'].copy()
    return out


def plot_clean_calibration(points_df: pd.DataFrame, slopes_df: pd.DataFrame, out_path: Path | None = None) -> None:
    plot_points = filtered_no_guidance(points_df)
    plot_slopes = filtered_no_guidance(slopes_df)

    fig, axes = plt.subplots(2, 3, figsize=(18.8, 10.2), constrained_layout=False)
    for ax, param in zip(axes.ravel(), PARAM_ORDER):
        sub_param = plot_points[plot_points['parameter'] == param].copy()
        if sub_param.empty:
            ax.set_visible(False)
            continue
        lo = float(min(sub_param['theta_in'].min(), sub_param['theta_rec_q16'].min()))
        hi = float(max(sub_param['theta_in'].max(), sub_param['theta_rec_q84'].max()))
        pad = 0.08 * max(hi - lo, 1e-6)
        lo, hi = lo - pad, hi + pad
        ax.plot([lo, hi], [lo, hi], color='0.25', lw=2.0, ls='--', label='ideal recovery')

        for regime in ['memorization', 'generalization']:
            sub = sub_param[sub_param['regime'] == regime].sort_values('theta_in')
            if sub.empty:
                continue
            y = sub['theta_rec_median'].to_numpy(float)
            yerr = np.vstack([
                np.maximum(y - sub['theta_rec_q16'].to_numpy(float), 0.0),
                np.maximum(sub['theta_rec_q84'].to_numpy(float) - y, 0.0),
            ])
            ax.errorbar(
                sub['theta_in'], y, yerr=yerr,
                fmt=REGIME_MARKERS.get(regime, 'o'), ms=7.4, lw=1.8, capsize=3.0,
                color=REGIME_COLORS.get(regime, 'black'), ecolor=REGIME_COLORS.get(regime, 'black'),
                alpha=0.86, markeredgecolor='white', markeredgewidth=0.7,
                label=REGIME_LABELS.get(regime, regime),
            )
            fit = plot_slopes[(plot_slopes['parameter'] == param) & (plot_slopes['regime'] == regime)]
            if not fit.empty:
                slope = float(fit['slope'].iloc[0])
                intercept = float(fit['intercept'].iloc[0])
                ax.plot([lo, hi], [slope * lo + intercept, slope * hi + intercept],
                        color=REGIME_COLORS.get(regime, 'black'), lw=3.0)
                short = 'mem.' if regime == 'memorization' else 'gen.'
                ax.text(0.035, 0.93 if regime == 'memorization' else 0.84,
                        f"{short} slope = {slope:.2f}",
                        color=REGIME_COLORS[regime], transform=ax.transAxes,
                        fontsize=12.0, fontweight='bold', ha='left', va='top')

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(PARAM_LABELS.get(param, param), pad=8)
        ax.set_xlabel('Requested input value')
        ax.set_ylabel('Recovered value')
        ax.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    fig.legend(dedup.values(), dedup.keys(), loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.945), fontsize=15)
    fig.suptitle('Continuous HI cosmology calibration', y=0.99, fontsize=25)
    fig.text(0.5, 0.952, 'Generated fields encoded back to cosmology with the best VGG probe',
             ha='center', va='center', fontsize=14.5, color='0.25')
    fig.subplots_adjust(left=0.055, right=0.99, bottom=0.07, top=0.84, wspace=0.25, hspace=0.34)
    if out_path is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, bbox_inches='tight')
        print('wrote', out_path)
    plt.show()


def plot_omega_m_poster_calibration(points_df: pd.DataFrame, slopes_df: pd.DataFrame, out_path: Path | None = None) -> None:
    plot_points = filtered_no_guidance(points_df)
    plot_slopes = filtered_no_guidance(slopes_df)
    param = 'Omega_m'
    sub_param = plot_points[plot_points['parameter'] == param].copy()
    if sub_param.empty:
        raise ValueError('No Omega_m rows found in calibration points table.')

    fig, ax = plt.subplots(figsize=(7.4, 5.35), constrained_layout=True)
    lo = float(min(sub_param['theta_in'].min(), sub_param['theta_rec_q16'].min()))
    hi = float(max(sub_param['theta_in'].max(), sub_param['theta_rec_q84'].max()))
    pad = 0.09 * max(hi - lo, 1e-6)
    lo, hi = lo - pad, hi + pad
    ax.plot([lo, hi], [lo, hi], color='0.25', lw=2.25, ls='--', label='ideal recovery', zorder=1)

    for regime in ['memorization', 'generalization']:
        sub = sub_param[sub_param['regime'] == regime].sort_values('theta_in')
        if sub.empty:
            continue
        y = sub['theta_rec_median'].to_numpy(float)
        yerr = np.vstack([
            np.maximum(y - sub['theta_rec_q16'].to_numpy(float), 0.0),
            np.maximum(sub['theta_rec_q84'].to_numpy(float) - y, 0.0),
        ])
        fit = plot_slopes[(plot_slopes['parameter'] == param) & (plot_slopes['regime'] == regime)]
        slope = float(fit['slope'].iloc[0]) if not fit.empty else np.nan
        intercept = float(fit['intercept'].iloc[0]) if not fit.empty else np.nan
        label = REGIME_LABELS[regime]
        if np.isfinite(slope):
            label += f', slope={slope:.2f}'

        ax.errorbar(
            sub['theta_in'], y, yerr=yerr,
            fmt=REGIME_MARKERS[regime], ms=8.2, lw=2.0, capsize=3.2, capthick=1.35,
            color=REGIME_COLORS[regime], ecolor=REGIME_COLORS[regime],
            markeredgecolor='white', markeredgewidth=0.8, alpha=0.88,
            label=label, zorder=3,
        )
        if np.isfinite(slope):
            xs = np.array([lo, hi])
            ax.plot(xs, slope * xs + intercept, color=REGIME_COLORS[regime], lw=3.4, zorder=2)

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(r'Requested $\Omega_m$', fontsize=19)
    ax.set_ylabel(r'Recovered $\Omega_m$', fontsize=19)
    ax.set_title(r'Large-data samples track the requested $\Omega_m$', fontsize=20, pad=15)
    ax.tick_params(labelsize=16)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=1, frameon=False, fontsize=13.5)
    ax.text(0.04, 0.955, 'Probe: frozen VGG16 + avg+max + MLP(1024,512,256)',
            transform=ax.transAxes, ha='left', va='top', fontsize=12.5, color='0.28')

    if out_path is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, bbox_inches='tight', dpi=300)
        print('wrote', out_path)
    plt.show()


clean_plot_path = CAL_DIR / 'bias_probe_calibration_recovered_vs_input_clean.png'
plot_clean_calibration(points, slopes, clean_plot_path)

omega_poster_path = CAL_DIR / 'bias_probe_omega_m_best_vgg_poster.png'
plot_omega_m_poster_calibration(points, slopes, omega_poster_path)

## What The Error Bars Mean

For each held-out input cosmology, the sampler generates `K` different HI fields using different noise seeds. Each field is passed through the VGG cosmology probe. The plotted marker is the median recovered value across those `K` generated samples. The vertical error bar is the 16th to 84th percentile range across those same `K` samples.

So the error bar is not the uncertainty of the input cosmology, and it is not the encoder validation error. It is the stochastic spread of generated samples at fixed input cosmology after passing through the encoder.

If the memorization model has smaller error bars, that usually means it produces less seed-to-seed variation at fixed conditioning. That can happen if it is more collapsed or more tied to memorized training-like fields. Smaller error bars do not automatically mean better calibration; the median location and the fitted slope matter more for bias.

In [ ]:
err = points.copy()
err['half_width_16_84'] = 0.5 * (err['theta_rec_q84'] - err['theta_rec_q16'])
err_summary = (
    err.groupby(['regime', 'dataset_size', 'parameter'], as_index=False)
       .agg(mean_half_width=('half_width_16_84', 'mean'), median_half_width=('half_width_16_84', 'median'))
       .sort_values(['parameter', 'dataset_size'])
)
display(with_parameter_labels(err_summary).round(4))

fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
pivot = err_summary.pivot(index='parameter', columns='regime', values='mean_half_width').reindex(PARAM_ORDER)
x = np.arange(len(pivot))
width = 0.36
ax.bar(x - width/2, pivot.get('memorization'), width, color=REGIME_COLORS['memorization'], label='memorization (N=128)')
ax.bar(x + width/2, pivot.get('generalization'), width, color=REGIME_COLORS['generalization'], label='generalization (N=16,384)')
ax.set_xticks(x, [PARAM_LABELS.get(p, p) for p in pivot.index])
ax.set_ylabel('mean 16-84% half-width')
ax.set_title('Generated-sample spread at fixed input cosmology')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
spread_path = CAL_DIR / 'bias_probe_errorbar_spread_summary.png'
fig.savefig(spread_path, bbox_inches='tight')
print('wrote', spread_path)
plt.show()

## Chi-Square Coverage Check

Nick's suggested coverage diagnostic is to compare the recovered median to the true input in units of the plotted error bar:

```text
chi2 = (theta_true - theta_predict)^2 / errorbar^2
```

Here `theta_predict` is the median recovered parameter for one held-out cosmology, and `errorbar` comes from the 16/84 percentile spread across the generated samples at that same input cosmology. The exact number of generated samples per point is stored in the `n_samples` column printed above.

The dashed `chi^2_1` curve is the ideal reference for one normalized Gaussian residual. If `z = (theta_rec - theta_true) / sigma` is distributed like a standard normal, then `z^2` follows a chi-square distribution with one degree of freedom, written `chi^2_1`. For that ideal distribution, the mean is `1.0`, but the median is only about `0.455`.

How to read it:

- curve far to the right of `chi^2_1`: truth is often many error bars away, so coverage is bad or the model is biased;
- curve close to `chi^2_1`: error bars and median errors are roughly calibrated;
- curve far to the left: error bars are probably too wide for the observed median errors.

Because our error bars are generated-sample spread, not formal uncertainty on the median or encoder uncertainty, this is a coverage stress test rather than a complete likelihood calibration test. Still, it is useful: large `chi2` means the true input lies far outside the generated recovered-parameter spread; very small `chi2` means the error bars may be broad relative to the bias.

In [ ]:
chi = points.copy()
chi['err_low'] = chi['theta_rec_median'] - chi['theta_rec_q16']
chi['err_high'] = chi['theta_rec_q84'] - chi['theta_rec_median']
chi['err_symmetric_16_84'] = 0.5 * (chi['err_low'] + chi['err_high'])
# Use the side of the asymmetric 16/84 error bar that points toward the truth.
chi['err_directional_16_84'] = np.where(
    chi['theta_in'] < chi['theta_rec_median'],
    chi['err_low'],
    chi['err_high'],
)
chi['delta'] = chi['theta_rec_median'] - chi['theta_in']
chi['z_directional'] = chi['delta'] / chi['err_directional_16_84'].replace(0, np.nan)
chi['chi2_directional'] = chi['z_directional'] ** 2
chi['z_symmetric'] = chi['delta'] / chi['err_symmetric_16_84'].replace(0, np.nan)
chi['chi2_symmetric'] = chi['z_symmetric'] ** 2
chi = chi.replace([np.inf, -np.inf], np.nan)

chi_points_out = CAL_DIR / 'bias_probe_chi2_points.csv'
chi.to_csv(chi_points_out, index=False)
print('wrote', chi_points_out)

chi_summary = (
    chi.groupby(['regime', 'dataset_size', 'parameter'], as_index=False)
       .agg(
           n_points=('chi2_directional', 'count'),
           mean_chi2=('chi2_directional', 'mean'),
           median_chi2=('chi2_directional', 'median'),
           p68_covered=('chi2_directional', lambda s: float(np.mean(s.dropna().to_numpy() <= 1.0)) if len(s.dropna()) else np.nan),
           p95_covered=('chi2_directional', lambda s: float(np.mean(s.dropna().to_numpy() <= 4.0)) if len(s.dropna()) else np.nan),
           mean_abs_z=('z_directional', lambda s: float(np.nanmean(np.abs(np.asarray(s))))),
       )
       .sort_values(['parameter', 'dataset_size'])
)
chi_summary_out = CAL_DIR / 'bias_probe_chi2_summary.csv'
chi_summary.to_csv(chi_summary_out, index=False)
print('wrote', chi_summary_out)
display(with_parameter_labels(chi_summary).round(4))

finite_chi = chi[np.isfinite(chi['chi2_directional']) & (chi['chi2_directional'] >= 0)].copy()
if finite_chi.empty:
    print('No finite chi-square values to plot.')
else:
    # The memorization model can be hundreds in chi-square, so use a log-x CDF rather than a linear histogram.
    positive = finite_chi['chi2_directional'].to_numpy(float)
    x_min = max(1.0e-3, float(np.nanquantile(positive[positive > 0], 0.01)) if np.any(positive > 0) else 1.0e-3)
    x_max = min(max(float(np.nanquantile(positive, 0.995)), 10.0), 1.0e4)
    xgrid = np.logspace(np.log10(x_min), np.log10(x_max), 600)

    def chi2_1_cdf(x: np.ndarray) -> np.ndarray:
        # CDF of chi-square with one degree of freedom: erf(sqrt(x / 2)).
        from math import erf
        return np.asarray([erf(float(np.sqrt(v / 2.0))) for v in x], dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(16.2, 5.8), constrained_layout=True)

    ax = axes[0]
    for regime in ['memorization', 'generalization']:
        vals = np.sort(finite_chi.loc[finite_chi['regime'] == regime, 'chi2_directional'].to_numpy(float))
        if len(vals) == 0:
            continue
        y = np.arange(1, len(vals) + 1, dtype=float) / len(vals)
        vals_plot = np.clip(vals, x_min, None)
        ax.step(vals_plot, y, where='post', lw=3.0, color=REGIME_COLORS.get(regime, 'black'), label=REGIME_LABELS.get(regime, regime))
    ax.plot(xgrid, chi2_1_cdf(xgrid), color='0.2', ls='--', lw=2.2, label=r'$\chi^2_1$ reference')
    ax.axvline(1.0, color='0.35', ls=':', lw=1.8)
    ax.axvline(4.0, color='0.35', ls=':', lw=1.8)
    ax.set_xscale('log')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel(r'$(\theta_\mathrm{rec} - \theta_\mathrm{true})^2 / \sigma^2$')
    ax.set_ylabel('Empirical CDF')
    ax.set_title('Coverage over held-out cosmology points')
    ax.grid(alpha=0.18, which='both')
    ax.legend(frameon=False, loc='lower right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax = axes[1]
    summary_pivot = chi_summary.pivot(index='parameter', columns='regime', values='median_chi2').reindex(PARAM_ORDER)
    x = np.arange(len(summary_pivot))
    width = 0.36
    ax.bar(x - width/2, summary_pivot.get('memorization'), width, color=REGIME_COLORS['memorization'], label=REGIME_LABELS['memorization'])
    ax.bar(x + width/2, summary_pivot.get('generalization'), width, color=REGIME_COLORS['generalization'], label=REGIME_LABELS['generalization'])
    chi2_median_ref = 0.454936423119572
    ax.axhline(chi2_median_ref, color='0.2', ls='--', lw=2.0, label=r'$\chi^2_1$ median = 0.455')
    ax.axhline(1.0, color='0.25', ls='-.', lw=1.8, label=r'$\chi^2_1$ mean = 1')
    ax.axhline(4.0, color='0.35', ls=':', lw=1.8, label='2σ reference')
    ax.set_yscale('log')
    ax.set_xticks(x, [PARAM_LABELS.get(p, p) for p in summary_pivot.index])
    ax.set_ylabel('Median chi-square')
    ax.set_title('Median chi-square by parameter')
    ax.grid(axis='y', alpha=0.18, which='both')
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.suptitle('Coverage check from recovered cosmology error bars', y=1.04)
    chi_plot_path = CAL_DIR / 'bias_probe_chi2_coverage.png'
    fig.savefig(chi_plot_path, bbox_inches='tight')
    print('wrote', chi_plot_path)
    plt.show()


## Generated Field Fidelity: One-Point PDF And Auto-Power

Before interpreting the cosmology calibration slopes, check whether the generated HI fields match simple real-field statistics.

This section compares the generated held-out samples against real held-out CAMELS HI slices using the same log+tanh normalization as training. The comparison is distribution-level: it does not require generated fields to match the exact real slice index.

- **One-point PDF:** checks the normalized pixel-value distribution.
- **Auto-power `P(k)`:** checks spatial structure. Ratios near 1 are better.

If `N=16,384` has poor one-point/P(k), then the calibration plot is probably contaminated by undertraining. If one-point/P(k) look reasonable but the calibration slope is still far below 1, then the model may be generating realistic HI while not using the input cosmology strongly enough.


In [ ]:
from simdiff_eval.metrics import batch_power_spectra, power_spectrum_summary

SAMPLE_DIR = RESULT_DIR / 'samples'
PK_NBINS = int(os.environ.get('BIAS_PROBE_PK_NBINS', 30))
MAX_PK_IMAGES = int(os.environ.get('BIAS_PROBE_MAX_PK_IMAGES', 2048))
ONEPOINT_BINS = int(os.environ.get('BIAS_PROBE_ONEPOINT_BINS', 220))

REGIME_SHORT = {'memorization': 'N=128', 'generalization': 'N=16,384'}


def resolve_project_path(value: str | Path) -> Path:
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_DIR / path


def guidance_label_from_row(row: dict) -> str:
    return 'noguidance'


def sample_path_for_row(row: dict, seed: int = 123) -> Path:
    k = int(row.get('heldout_samples_per_cosmology', 64))
    raw = str(row['sample_path'])
    rel = raw.format(seed=seed, sample_label='dpm50', k=k, guidance=guidance_label_from_row(row))
    return PROJECT_DIR / rel


def tanh_normalize_logged(log_images: np.ndarray, norm: dict) -> np.ndarray:
    center = np.float32(norm['center'])
    xmax = np.float32(norm['xmax'])
    alpha = np.float32(norm.get('alpha', 0.8))
    beta = np.float32(norm.get('beta', 10.0))
    gamma = np.float32(norm.get('gamma', 1.0))
    delta = np.float32(norm.get('delta', 1.0))
    sigma = np.float32(norm.get('sigma', 1.5))
    x = (log_images.astype(np.float32, copy=False) - center) / xmax
    pos = alpha * np.tanh((gamma * x) / alpha)
    neg = beta * np.tanh((delta * x) / beta)
    return (np.where(x >= 0, pos, neg) * sigma).astype(np.float32, copy=False)


def preprocess_real_hi(raw_images: np.ndarray, norm: dict) -> np.ndarray:
    logged = np.log(np.maximum(raw_images.astype(np.float32, copy=False), np.float32(1.0e-30)))
    return tanh_normalize_logged(logged, norm)


def load_generated_samples(row: dict) -> np.ndarray:
    path = sample_path_for_row(row)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=True) as data:
        samples_arr = data['samples'].astype(np.float32)
    if samples_arr.ndim == 3:
        samples_arr = samples_arr[:, None]
    return samples_arr


def load_heldout_real_reference(row: dict) -> np.ndarray:
    k = int(row.get('heldout_samples_per_cosmology', 64))
    heldout_indices = np.loadtxt(resolve_project_path(row['heldout_indices_path']), dtype=np.int64)
    grid_path = resolve_project_path(row['data_path'])
    grid = np.load(grid_path, mmap_mode='r')
    z_count = int(grid.shape[1])
    z_indices = np.linspace(0, z_count - 1, k, dtype=np.int64)
    out = np.empty((len(heldout_indices) * len(z_indices), 1, grid.shape[-2], grid.shape[-1]), dtype=np.float32)
    n = 0
    for sim_idx in heldout_indices:
        for z_idx in z_indices:
            out[n, 0] = np.asarray(grid[int(sim_idx), int(z_idx)], dtype=np.float32)
            n += 1
    return preprocess_real_hi(out, row['normalization'])


def histogram_l1(a: np.ndarray, b: np.ndarray, bins: np.ndarray) -> float:
    ha, _ = np.histogram(np.asarray(a).ravel(), bins=bins, density=True)
    hb, _ = np.histogram(np.asarray(b).ravel(), bins=bins, density=True)
    return float(np.sum(np.abs(ha - hb) * np.diff(bins)))


def evenly_limit(arr: np.ndarray, n: int) -> np.ndarray:
    if len(arr) <= n:
        return arr
    idx = np.linspace(0, len(arr) - 1, int(n), dtype=np.int64)
    return arr[idx]


def density_curve(values: np.ndarray, bins: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    hist, edges = np.histogram(np.asarray(values).ravel(), bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist


main_rows = [row for row in manifest_rows if row.get('run_name') in MAIN_JOB_INFO]
if not main_rows:
    raise RuntimeError('No main bias-probe rows found in manifest. Run the prepare step or pull the latest local manifest on Great Lakes.')

physics_cache = {}
physics_rows = []
for row in sorted(main_rows, key=lambda r: int(r['dataset_size'])):
    regime = row['regime']
    generated = load_generated_samples(row)
    real = load_heldout_real_reference(row)
    if real.shape != generated.shape:
        print('shape note:', row['run_name'], 'real', real.shape, 'generated', generated.shape)

    combined = np.concatenate([real.reshape(-1), generated.reshape(-1)])
    lo, hi = np.quantile(combined[np.isfinite(combined)], [0.001, 0.999])
    pad = 0.04 * max(float(hi - lo), 1e-6)
    bins = np.linspace(float(lo - pad), float(hi + pad), ONEPOINT_BINS)

    real_pk = evenly_limit(real, MAX_PK_IMAGES)
    gen_pk = evenly_limit(generated, MAX_PK_IMAGES)
    pk_real, kbins = batch_power_spectra(real_pk, nbins=PK_NBINS)
    pk_gen, _ = batch_power_spectra(gen_pk, nbins=PK_NBINS)
    real_mean_pk = np.nanmean(pk_real, axis=0)
    gen_mean_pk = np.nanmean(pk_gen, axis=0)
    ratio = gen_mean_pk / np.clip(real_mean_pk, 1e-30, None)

    pk_summary = power_spectrum_summary(real_pk, gen_pk, nbins=PK_NBINS)
    row_summary = {
        'regime': regime,
        'run_name': row['run_name'],
        'dataset_size': int(row['dataset_size']),
        'actual_updates': int(row['actual_updates']),
        'n_real': int(len(real)),
        'n_generated': int(len(generated)),
        'n_pk_used': int(len(gen_pk)),
        'hist_l1': histogram_l1(real, generated, bins),
        'real_mean': float(np.mean(real)),
        'generated_mean': float(np.mean(generated)),
        'real_std': float(np.std(real)),
        'generated_std': float(np.std(generated)),
        **pk_summary,
    }
    physics_rows.append(row_summary)
    physics_cache[regime] = {
        'row': row,
        'real': real,
        'generated': generated,
        'bins': bins,
        'kbins': kbins,
        'pk_real': pk_real,
        'pk_gen': pk_gen,
        'real_mean_pk': real_mean_pk,
        'gen_mean_pk': gen_mean_pk,
        'ratio': ratio,
    }

physics_summary = pd.DataFrame(physics_rows).sort_values('dataset_size')
display(physics_summary.round(5))
metrics_out = CAL_DIR / 'bias_probe_onepoint_pk_metrics.csv'
physics_summary.to_csv(metrics_out, index=False)
print('wrote', metrics_out)

# One-point PDFs.
fig, axes = plt.subplots(1, len(physics_cache), figsize=(7.6 * len(physics_cache), 5.6), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, regime in zip(axes, ['memorization', 'generalization']):
    if regime not in physics_cache:
        ax.set_visible(False)
        continue
    info = physics_cache[regime]
    bins = info['bins']
    x_real, y_real = density_curve(info['real'], bins)
    x_gen, y_gen = density_curve(info['generated'], bins)
    ax.step(x_real, y_real, where='mid', color='0.15', lw=2.8, label='real held-out')
    ax.step(x_gen, y_gen, where='mid', color=REGIME_COLORS.get(regime, 'tab:blue'), lw=2.8, label='generated')
    ax.set_yscale('log')
    ax.set_xlabel('Normalized HI value')
    ax.set_ylabel('Density')
    ax.set_title(f"{REGIME_LABELS.get(regime, regime)} one-point PDF")
    ax.grid(alpha=0.18)
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
onepoint_path = CAL_DIR / 'bias_probe_onepoint_pdf.png'
fig.savefig(onepoint_path, bbox_inches='tight')
print('wrote', onepoint_path)
plt.show()

# Auto-power P(k).
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8), constrained_layout=True)
ax = axes[0]
plotted_real = False
for regime in ['memorization', 'generalization']:
    if regime not in physics_cache:
        continue
    info = physics_cache[regime]
    k = info['kbins']
    real_mean = info['real_mean_pk']
    real_lo, real_hi = np.nanquantile(info['pk_real'], [0.16, 0.84], axis=0)
    if not plotted_real:
        ax.plot(k, real_mean, color='0.15', lw=3.0, label='real held-out')
        ax.fill_between(k, real_lo, real_hi, color='0.65', alpha=0.22, linewidth=0)
        plotted_real = True
    ax.plot(k, info['gen_mean_pk'], color=REGIME_COLORS.get(regime, 'tab:blue'), lw=2.8, label=f"generated {REGIME_SHORT.get(regime, regime)}")
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Spatial frequency k')
ax.set_ylabel('Mean P(k)')
ax.set_title('Auto-power spectrum')
ax.grid(alpha=0.18)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
for regime in ['memorization', 'generalization']:
    if regime not in physics_cache:
        continue
    info = physics_cache[regime]
    ax.plot(info['kbins'], info['ratio'], color=REGIME_COLORS.get(regime, 'tab:blue'), lw=2.8, marker='o', ms=4.5, label=REGIME_LABELS.get(regime, regime))
ax.axhline(1.0, color='0.15', ls='--', lw=2.0)
ax.set_xscale('log')
ax.set_xlabel('Spatial frequency k')
ax.set_ylabel('Generated / real P(k)')
ax.set_title('Auto-power ratio')
ax.grid(alpha=0.18)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
pk_path = CAL_DIR / 'bias_probe_pk_ratio.png'
fig.savefig(pk_path, bbox_inches='tight')
print('wrote', pk_path)
plt.show()


## Nearest Training-Slice Check

The one-point and `P(k)` plots compare generated samples to a broad held-out real distribution. That is the right fidelity check, but it is not the best memorization check for the tiny `N=128` run. A model can fail to match the held-out distribution precisely because it is memorizing a small training subset.

This section asks the direct memorization question: how close are generated samples to the actual training slices used by that run?

It computes two nearest-training diagnostics:

- Downsampled normalized-pixel L2 distance: fast distribution-level check. Lower distance means closer to a training slice.
- Full 128x128 normalized-pixel cosine similarity: stricter visual-copy check. Identical images have cosine = 1. If the `N=128` model cannot get close to cosine 1 for its nearest training slice, then it is not cleanly memorizing exact training images; that points to undertraining, imperfect memorization, or sampler/model mismatch.

The comparison baselines are:

- generated sample -> nearest training slice,
- held-out real slice -> nearest training slice,
- training slice -> nearest other training slice.

For a strong memorization signal, generated-to-training cosine should be higher than held-out-real-to-training cosine, and the closest generated/training image pairs should look visibly similar. For a generalizing model, generated-to-training similarity should look closer to held-out-real-to-training similarity.


In [ ]:
NN_DOWNSAMPLE = int(os.environ.get('BIAS_PROBE_NN_DOWNSAMPLE', 16))
NN_MAX_QUERY = int(os.environ.get('BIAS_PROBE_NN_MAX_QUERY', 512))
NN_BATCH = int(os.environ.get('BIAS_PROBE_NN_BATCH', 128))
NN_FEATURE_BATCH = int(os.environ.get('BIAS_PROBE_NN_FEATURE_BATCH', 512))
NN_COS_BATCH = int(os.environ.get('BIAS_PROBE_NN_COS_BATCH', NN_FEATURE_BATCH))
NN_EXAMPLES = int(os.environ.get('BIAS_PROBE_NN_EXAMPLES', 4))


def average_pool_features(images: np.ndarray, out_hw: int = NN_DOWNSAMPLE) -> np.ndarray:
    arr = np.asarray(images, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W), got {arr.shape}')
    h, w = arr.shape[-2:]
    if h % out_hw != 0 or w % out_hw != 0:
        raise ValueError(f'Cannot average-pool {h}x{w} to {out_hw}x{out_hw}')
    by = h // out_hw
    bx = w // out_hw
    pooled = arr.reshape(len(arr), 1, out_hw, by, out_hw, bx).mean(axis=(3, 5))
    return pooled.reshape(len(arr), -1).astype(np.float32, copy=False)


def feature_normalized_l2_nn(query_features: np.ndarray, ref_features: np.ndarray, *, batch_size: int = NN_BATCH, exclude_self: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    q = np.asarray(query_features, dtype=np.float32)
    r = np.asarray(ref_features, dtype=np.float32)
    r_norm = np.sum(r * r, axis=1)[None, :]
    dim = max(1, q.shape[1])
    best_dist = []
    best_idx = []
    for start in range(0, len(q), batch_size):
        stop = min(start + batch_size, len(q))
        qb = q[start:stop]
        d2 = np.sum(qb * qb, axis=1)[:, None] + r_norm - 2.0 * (qb @ r.T)
        if exclude_self is not None:
            for local, ref_i in enumerate(exclude_self[start:stop]):
                if 0 <= int(ref_i) < d2.shape[1]:
                    d2[local, int(ref_i)] = np.inf
        idx = np.argmin(d2, axis=1)
        dist = np.sqrt(np.maximum(d2[np.arange(stop - start), idx], 0.0) / dim)
        best_dist.append(dist.astype(np.float32))
        best_idx.append(idx.astype(np.int64))
    return np.concatenate(best_dist), np.concatenate(best_idx)


def flatten_images_for_cosine(images: np.ndarray) -> np.ndarray:
    arr = np.asarray(images, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W), got {arr.shape}')
    flat = arr.reshape(len(arr), -1).astype(np.float32, copy=False)
    norm = np.linalg.norm(flat, axis=1, keepdims=True)
    norm = np.maximum(norm, 1e-12)
    return flat / norm


def pixel_cosine_nn_to_training(query_images: np.ndarray, row: dict, *, batch_size: int = NN_COS_BATCH, exclude_self: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    # Full-pixel cosine, streamed over training batches. Identical images give cosine=1.
    q = flatten_images_for_cosine(query_images)
    train_path = resolve_project_path(row['prepared_image_path'])
    raw = np.load(train_path, mmap_mode='r')
    best_cos = np.full(len(q), -np.inf, dtype=np.float32)
    best_idx = np.zeros(len(q), dtype=np.int64)
    for start in range(0, len(raw), batch_size):
        stop = min(start + batch_size, len(raw))
        train_batch = preprocess_real_hi(np.asarray(raw[start:stop], dtype=np.float32), row['normalization'])
        r = flatten_images_for_cosine(train_batch)
        sims = q @ r.T
        if exclude_self is not None:
            for local, ref_i in enumerate(np.asarray(exclude_self, dtype=np.int64)):
                if start <= int(ref_i) < stop:
                    sims[local, int(ref_i) - start] = -np.inf
        local_idx = np.argmax(sims, axis=1)
        local_best = sims[np.arange(len(q)), local_idx]
        update = local_best > best_cos
        best_cos[update] = local_best[update].astype(np.float32)
        best_idx[update] = (start + local_idx[update]).astype(np.int64)
    return best_cos, best_idx


def selected_query_indices(n: int, max_query: int = NN_MAX_QUERY) -> np.ndarray:
    if n <= max_query:
        return np.arange(n, dtype=np.int64)
    return np.linspace(0, n - 1, int(max_query), dtype=np.int64)


def training_feature_matrix(row: dict) -> np.ndarray:
    train_path = resolve_project_path(row['prepared_image_path'])
    raw = np.load(train_path, mmap_mode='r')
    features = []
    for start in range(0, len(raw), NN_FEATURE_BATCH):
        stop = min(start + NN_FEATURE_BATCH, len(raw))
        batch = preprocess_real_hi(np.asarray(raw[start:stop], dtype=np.float32), row['normalization'])
        features.append(average_pool_features(batch, NN_DOWNSAMPLE))
    return np.concatenate(features, axis=0)


def training_images_by_index(row: dict, indices: np.ndarray) -> np.ndarray:
    train_path = resolve_project_path(row['prepared_image_path'])
    raw = np.load(train_path, mmap_mode='r')
    arr = np.asarray(raw[np.asarray(indices, dtype=np.int64)], dtype=np.float32)
    return preprocess_real_hi(arr, row['normalization'])


def training_slice_metadata(row: dict) -> tuple[pd.DataFrame, np.ndarray]:
    pairs = pd.read_csv(resolve_project_path(row['selected_pairs_path']))
    raw_params = np.load(resolve_project_path(row['train_raw_params_path'])).astype(np.float32)
    if len(pairs) != len(raw_params):
        raise ValueError(f"metadata length mismatch for {row['run_name']}: {len(pairs)} pairs vs {len(raw_params)} params")
    return pairs, raw_params


nn_rows = []
nn_cos_rows = []
nn_pair_rows = []
nn_cache = {}
for row in sorted(main_rows, key=lambda r: int(r['dataset_size'])):
    regime = row['regime']
    generated = load_generated_samples(row)
    heldout_real = load_heldout_real_reference(row)
    train_features = training_feature_matrix(row)
    train_pairs, train_raw_params = training_slice_metadata(row)
    heldout_indices = np.loadtxt(resolve_project_path(row['heldout_indices_path']), dtype=np.int64)
    k_per_cosmology = int(row.get('heldout_samples_per_cosmology', 64))

    gen_query_idx = selected_query_indices(len(generated), NN_MAX_QUERY)
    real_query_idx = selected_query_indices(len(heldout_real), NN_MAX_QUERY)
    train_query_idx = selected_query_indices(len(train_features), NN_MAX_QUERY)

    gen_features = average_pool_features(generated[gen_query_idx], NN_DOWNSAMPLE)
    real_features = average_pool_features(heldout_real[real_query_idx], NN_DOWNSAMPLE)
    train_query_features = train_features[train_query_idx]
    train_query_images = training_images_by_index(row, train_query_idx)

    gen_dist, gen_nn = feature_normalized_l2_nn(gen_features, train_features)
    real_dist, real_nn = feature_normalized_l2_nn(real_features, train_features)
    train_dist, train_nn = feature_normalized_l2_nn(train_query_features, train_features, exclude_self=train_query_idx)

    gen_cos, gen_cos_nn = pixel_cosine_nn_to_training(generated[gen_query_idx], row)
    real_cos, real_cos_nn = pixel_cosine_nn_to_training(heldout_real[real_query_idx], row)
    train_cos, train_cos_nn = pixel_cosine_nn_to_training(train_query_images, row, exclude_self=train_query_idx)

    for source, dists in [('generated_to_train', gen_dist), ('heldout_real_to_train', real_dist), ('train_to_train_other', train_dist)]:
        nn_rows.append({
            'regime': regime,
            'run_name': row['run_name'],
            'dataset_size': int(row['dataset_size']),
            'source': source,
            'n_query': int(len(dists)),
            'feature_downsample_hw': int(NN_DOWNSAMPLE),
            'median_nn_distance': float(np.median(dists)),
            'mean_nn_distance': float(np.mean(dists)),
            'q10_nn_distance': float(np.quantile(dists, 0.10)),
            'q90_nn_distance': float(np.quantile(dists, 0.90)),
        })

    for source, sims in [('generated_to_train', gen_cos), ('heldout_real_to_train', real_cos), ('train_to_train_other', train_cos)]:
        nn_cos_rows.append({
            'regime': regime,
            'run_name': row['run_name'],
            'dataset_size': int(row['dataset_size']),
            'source': source,
            'n_query': int(len(sims)),
            'feature': 'full_128x128_normalized_pixel_cosine',
            'median_cosine': float(np.median(sims)),
            'mean_cosine': float(np.mean(sims)),
            'q10_cosine': float(np.quantile(sims, 0.10)),
            'q90_cosine': float(np.quantile(sims, 0.90)),
            'max_cosine': float(np.max(sims)),
        })

    example_order = np.argsort(-gen_cos)[:max(1, NN_EXAMPLES)]
    example_gen_indices = gen_query_idx[example_order]
    example_train_indices = gen_cos_nn[example_order]
    nn_cache[regime] = {
        'row': row,
        'generated': generated,
        'gen_query_idx': gen_query_idx,
        'gen_dist': gen_dist,
        'real_dist': real_dist,
        'train_dist': train_dist,
        'gen_nn': gen_nn,
        'gen_cos': gen_cos,
        'real_cos': real_cos,
        'train_cos': train_cos,
        'gen_cos_nn': gen_cos_nn,
        'example_gen_indices': example_gen_indices,
        'example_train_indices': example_train_indices,
        'example_train_images': training_images_by_index(row, example_train_indices),
    }

    for local_i, gen_idx in enumerate(gen_query_idx):
        heldout_pos = int(gen_idx // k_per_cosmology)
        seed_index = int(gen_idx % k_per_cosmology)
        nearest_train_row = int(gen_nn[local_i])
        nearest_cos_train_row = int(gen_cos_nn[local_i])
        train_pair = train_pairs.iloc[nearest_train_row]
        cos_train_pair = train_pairs.iloc[nearest_cos_train_row]
        train_theta = train_raw_params[nearest_train_row]
        cos_train_theta = train_raw_params[nearest_cos_train_row]
        nn_pair_rows.append({
            'regime': regime,
            'run_name': row['run_name'],
            'dataset_size': int(row['dataset_size']),
            'generated_sample_index': int(gen_idx),
            'heldout_position': heldout_pos,
            'heldout_sim': int(heldout_indices[heldout_pos]),
            'seed_index': seed_index,
            'nearest_train_row': nearest_train_row,
            'nearest_train_sim': int(train_pair['simulation_index']),
            'nearest_train_z_index': int(train_pair['z_index']),
            'nearest_distance': float(gen_dist[local_i]),
            'nearest_cosine': float(gen_cos[local_i]),
            'nearest_cos_train_row': nearest_cos_train_row,
            'nearest_cos_train_sim': int(cos_train_pair['simulation_index']),
            'nearest_cos_train_z_index': int(cos_train_pair['z_index']),
            'train_Omega_m': float(train_theta[0]),
            'train_sigma_8': float(train_theta[1]),
            'cos_train_Omega_m': float(cos_train_theta[0]),
            'cos_train_sigma_8': float(cos_train_theta[1]),
        })

nn_summary = pd.DataFrame(nn_rows)
nn_cos_summary = pd.DataFrame(nn_cos_rows)
nn_cos_out = CAL_DIR / 'bias_probe_nearest_training_cosine_summary.csv'
nn_cos_summary.to_csv(nn_cos_out, index=False)
print('wrote', nn_cos_out)
display(nn_cos_summary.round(5))

nn_cos_pivot = nn_cos_summary.pivot_table(index=['regime', 'run_name', 'dataset_size'], columns='source', values='median_cosine').reset_index()
nn_cos_pivot['gen_minus_heldout_median_cosine'] = nn_cos_pivot['generated_to_train'] - nn_cos_pivot['heldout_real_to_train']
nn_cos_pivot['gen_minus_train_other_median_cosine'] = nn_cos_pivot['generated_to_train'] - nn_cos_pivot['train_to_train_other']
nn_cos_ratio_out = CAL_DIR / 'bias_probe_nearest_training_cosine_ratios.csv'
nn_cos_pivot.to_csv(nn_cos_ratio_out, index=False)
print('wrote', nn_cos_ratio_out)
display(nn_cos_pivot.round(5))

nn_summary_out = CAL_DIR / 'bias_probe_nearest_training_summary.csv'
nn_summary.to_csv(nn_summary_out, index=False)
print('wrote', nn_summary_out)
display(nn_summary.round(5))

nn_pivot = nn_summary.pivot_table(index=['regime', 'run_name', 'dataset_size'], columns='source', values='median_nn_distance').reset_index()
nn_pivot['gen_to_train_over_heldout_to_train'] = nn_pivot['generated_to_train'] / nn_pivot['heldout_real_to_train']
nn_pivot['gen_to_train_over_train_to_other_train'] = nn_pivot['generated_to_train'] / nn_pivot['train_to_train_other']
nn_ratio_out = CAL_DIR / 'bias_probe_nearest_training_ratios.csv'
nn_pivot.to_csv(nn_ratio_out, index=False)
print('wrote', nn_ratio_out)
display(nn_pivot.round(5))

nn_pairs = pd.DataFrame(nn_pair_rows).sort_values(['regime', 'nearest_distance'])
nn_pairs_out = CAL_DIR / 'bias_probe_nearest_training_pairs.csv'
nn_pairs.to_csv(nn_pairs_out, index=False)
print('wrote', nn_pairs_out)
display(nn_pairs.groupby('regime').head(8).round(5))

fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.7), constrained_layout=True)
for ax, regime in zip(axes, ['memorization', 'generalization']):
    if regime not in nn_cache:
        ax.set_visible(False)
        continue
    info = nn_cache[regime]
    for label, dists, color, ls in [
        ('generated -> train', info['gen_dist'], REGIME_COLORS.get(regime, 'tab:blue'), '-'),
        ('held-out real -> train', info['real_dist'], '0.2', '--'),
        ('train -> other train', info['train_dist'], '0.55', ':'),
    ]:
        vals = np.sort(np.asarray(dists, dtype=float))
        y = np.arange(1, len(vals) + 1, dtype=float) / len(vals)
        ax.step(vals, y, where='post', lw=2.7, color=color, ls=ls, label=label)
    med_gen = float(np.median(info['gen_dist']))
    med_real = float(np.median(info['real_dist']))
    ax.axvline(med_gen, color=REGIME_COLORS.get(regime, 'tab:blue'), lw=1.8, alpha=0.65)
    ax.axvline(med_real, color='0.2', lw=1.8, ls='--', alpha=0.65)
    ax.text(0.02, 0.06, f'gen / held-out median = {med_gen / med_real:.2f}', transform=ax.transAxes,
            ha='left', va='bottom', fontsize=11.5)
    ax.set_xlabel(f'Nearest training distance ({NN_DOWNSAMPLE}x{NN_DOWNSAMPLE} normalized pixels)')
    ax.set_ylabel('Empirical CDF')
    ax.set_title(REGIME_LABELS.get(regime, regime))
    ax.grid(alpha=0.18)
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Generated samples compared with closest training slices', y=1.04)
nn_dist_path = CAL_DIR / 'bias_probe_nearest_training_distance.png'
fig.savefig(nn_dist_path, bbox_inches='tight')
print('wrote', nn_dist_path)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.7), constrained_layout=True)
for ax, regime in zip(axes, ['memorization', 'generalization']):
    if regime not in nn_cache:
        ax.set_visible(False)
        continue
    info = nn_cache[regime]
    for label, sims, color, ls in [
        ('generated -> train', info['gen_cos'], REGIME_COLORS.get(regime, 'tab:blue'), '-'),
        ('held-out real -> train', info['real_cos'], '0.2', '--'),
        ('train -> other train', info['train_cos'], '0.55', ':'),
    ]:
        vals = np.sort(np.asarray(sims, dtype=float))
        y = np.arange(1, len(vals) + 1, dtype=float) / len(vals)
        ax.step(vals, y, where='post', lw=2.7, color=color, ls=ls, label=label)
    med_gen = float(np.median(info['gen_cos']))
    med_real = float(np.median(info['real_cos']))
    max_gen = float(np.max(info['gen_cos']))
    ax.axvline(med_gen, color=REGIME_COLORS.get(regime, 'tab:blue'), lw=1.8, alpha=0.65)
    ax.axvline(med_real, color='0.2', lw=1.8, ls='--', alpha=0.65)
    ax.text(0.02, 0.06, f'median gen cos={med_gen:.3f}\nmax gen cos={max_gen:.3f}', transform=ax.transAxes,
            ha='left', va='bottom', fontsize=11.5)
    ax.set_xlim(-0.05, 1.01)
    ax.set_xlabel('Cosine similarity to nearest training slice')
    ax.set_ylabel('Empirical CDF')
    ax.set_title(REGIME_LABELS.get(regime, regime))
    ax.grid(alpha=0.18)
    ax.legend(frameon=False, loc='upper left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Pixel-cosine similarity to closest training slices', y=1.04)
nn_cos_path = CAL_DIR / 'bias_probe_nearest_training_cosine.png'
fig.savefig(nn_cos_path, bbox_inches='tight')
print('wrote', nn_cos_path)
plt.show()

# Show the closest generated/training pairs. Use a shared color scale per pair so visual similarity is easy to judge.
cols = max(1, NN_EXAMPLES)
fig, axes = plt.subplots(4, cols, figsize=(3.0 * cols, 10.5), constrained_layout=True)
axes = np.asarray(axes)
for row_i, regime in enumerate(['memorization', 'generalization']):
    if regime not in nn_cache:
        axes[2 * row_i:2 * row_i + 2, :].ravel()[0].set_visible(False)
        continue
    info = nn_cache[regime]
    for j in range(cols):
        if j >= len(info['example_gen_indices']):
            axes[2 * row_i, j].set_visible(False)
            axes[2 * row_i + 1, j].set_visible(False)
            continue
        gen_idx = int(info['example_gen_indices'][j])
        train_idx = int(info['example_train_indices'][j])
        gen_img = info['generated'][gen_idx, 0]
        train_img = info['example_train_images'][j, 0]
        lo, hi = np.quantile(np.concatenate([gen_img.ravel(), train_img.ravel()]), [0.01, 0.99])
        axes[2 * row_i, j].imshow(gen_img, cmap='viridis', vmin=lo, vmax=hi)
        axes[2 * row_i + 1, j].imshow(train_img, cmap='viridis', vmin=lo, vmax=hi)
        local_pos = int(np.where(info['gen_query_idx'] == gen_idx)[0][0])
        axes[2 * row_i, j].set_title(f"{REGIME_SHORT.get(regime, regime)} gen #{gen_idx}\ncos={info['gen_cos'][local_pos]:.3f}")
        axes[2 * row_i + 1, j].set_title(f'nearest train #{train_idx}')
        axes[2 * row_i, j].axis('off')
        axes[2 * row_i + 1, j].axis('off')
fig.suptitle('Closest generated-to-training examples', y=1.02)
nn_examples_path = CAL_DIR / 'bias_probe_nearest_training_examples.png'
fig.savefig(nn_examples_path, bbox_inches='tight')
print('wrote', nn_examples_path)
plt.show()


## Optional Older Diagnostic: PCA Embedding Manifold

This section is not the default VGG cosmology probe. It is an older feature-space diagnostic using the frozen PCA basis from the PCA/Ridge encoder.

Nick's suggested diagnostic is different from the recovered-vs-input calibration plot. Here we take frozen PCA coefficients, reduce them to two dimensions with UMAP if available, otherwise t-SNE, and color the points by the true CAMELS parameters.

How to read this:

- If the real training/held-out points show smooth parameter gradients, then the PCA feature space contains cosmology information.
- If generated points lie on the same manifold but do not follow the input-parameter gradient, the diffusion model is likely weakly conditioned or undertrained.
- If generated points form a separate cloud or collapse into a small region, that is a stronger sign of sample-quality / training problems rather than only an encoder issue.
- The generated-point colors below are the input cosmologies requested from the diffusion model, not the recovered cosmologies.

In [ ]:

from train_nf_conditional_bias_encoder import load_pca

MANIFOLD_MAX_TRAIN = int(os.environ.get('BIAS_PROBE_MANIFOLD_MAX_TRAIN', 512))
MANIFOLD_MAX_HELDOUT = int(os.environ.get('BIAS_PROBE_MANIFOLD_MAX_HELDOUT', 512))
MANIFOLD_MAX_GENERATED = int(os.environ.get('BIAS_PROBE_MANIFOLD_MAX_GENERATED', 512))
MANIFOLD_BATCH = int(os.environ.get('BIAS_PROBE_MANIFOLD_BATCH', 512))
MANIFOLD_RANDOM_SEED = int(os.environ.get('BIAS_PROBE_MANIFOLD_SEED', 123))
MANIFOLD_METHOD_REQUEST = os.environ.get('BIAS_PROBE_MANIFOLD_METHOD', 'auto').strip().lower()


def load_pca_for_manifold():
    encoder_path = ENC_DIR / 'frozen_pca_ridge_encoder.npz'
    if not encoder_path.exists():
        raise FileNotFoundError(encoder_path)
    with np.load(encoder_path, allow_pickle=True) as data:
        basis_path = Path(str(data['pca_basis_path'].item()))
        if not basis_path.is_absolute():
            basis_path = PROJECT_DIR / basis_path
        rank = int(data['pca_rank'])
        ev = float(data['pca_explained_variance_sum'])
        sha = str(data['pca_basis_sha256'].item())
    pca = load_pca(basis_path)
    print(f'PCA manifold basis: {basis_path}')
    print(f'PCA rank={rank:,}, explained variance={ev:.4f}, sha256={sha[:12]}...')
    return pca, basis_path, rank, ev, sha


def evenly_spaced_indices(n: int, max_n: int) -> np.ndarray:
    if n <= max_n:
        return np.arange(n, dtype=np.int64)
    return np.linspace(0, n - 1, int(max_n), dtype=np.int64)


def load_generated_with_theta(row: dict) -> tuple[np.ndarray, np.ndarray]:
    path = sample_path_for_row(row)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=True) as data:
        images = data['samples'].astype(np.float32)
        theta = data['theta_raw'].astype(np.float32)
        k = int(data['samples_per_cosmology'])
    if images.ndim == 3:
        images = images[:, None]
    theta_rep = np.repeat(theta, k, axis=0)
    return images, theta_rep


def load_heldout_with_theta(row: dict) -> tuple[np.ndarray, np.ndarray]:
    images = load_heldout_real_reference(row)
    k = int(row.get('heldout_samples_per_cosmology', 64))
    theta_path = PROJECT_DIR / 'local' / SWEEP_NAME / 'heldout' / 'heldout_params_raw.npy'
    theta = np.load(theta_path).astype(np.float32)
    theta_rep = np.repeat(theta, k, axis=0)
    return images, theta_rep


def load_training_with_theta(row: dict) -> tuple[np.ndarray, np.ndarray]:
    train_path = resolve_project_path(row['prepared_image_path'])
    raw = np.load(train_path, mmap_mode='r')
    pairs, theta = training_slice_metadata(row)
    idx = evenly_spaced_indices(len(raw), MANIFOLD_MAX_TRAIN)
    images = preprocess_real_hi(np.asarray(raw[idx], dtype=np.float32), row['normalization'])
    return images, theta[idx].astype(np.float32)


def add_manifold_block(feature_rows, meta_rows, pca, *, row, source, images, theta, max_n):
    idx = evenly_spaced_indices(len(images), max_n)
    images = images[idx]
    theta = theta[idx]
    z = pca.transform(images, batch_size=MANIFOLD_BATCH)
    feature_rows.append(z)
    for local_i, theta_i in enumerate(theta):
        meta = {
            'regime': row['regime'],
            'run_name': row['run_name'],
            'dataset_size': int(row['dataset_size']),
            'source': source,
            'source_index': int(idx[local_i]),
        }
        for j, name in enumerate(PARAM_ORDER):
            meta[name] = float(theta_i[j])
        meta_rows.append(meta)


def reduce_manifold(features: np.ndarray) -> tuple[np.ndarray, str]:
    x = np.asarray(features, dtype=np.float32)
    x = x[:, :min(50, x.shape[1])]
    method = MANIFOLD_METHOD_REQUEST
    if method in {'auto', 'umap'}:
        try:
            import umap
            reducer = umap.UMAP(n_neighbors=30, min_dist=0.15, metric='euclidean', random_state=MANIFOLD_RANDOM_SEED)
            return reducer.fit_transform(x).astype(np.float32), 'UMAP'
        except Exception as exc:
            if method == 'umap':
                raise
            print('UMAP unavailable; falling back:', repr(exc))
    if method in {'auto', 'tsne', 't-sne'}:
        try:
            from sklearn.manifold import TSNE
            perplexity = min(30, max(5, (len(x) - 1) // 3))
            reducer = TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=perplexity, random_state=MANIFOLD_RANDOM_SEED)
            return reducer.fit_transform(x).astype(np.float32), f't-SNE perplexity={perplexity}'
        except Exception as exc:
            if method in {'tsne', 't-sne'}:
                raise
            print('t-SNE unavailable; falling back to PCA-2D:', repr(exc))
    from sklearn.decomposition import PCA
    coords = PCA(n_components=2, random_state=MANIFOLD_RANDOM_SEED).fit_transform(x)
    return coords.astype(np.float32), 'PCA-2D fallback'


pca_basis, pca_basis_path, pca_rank, pca_ev, pca_sha = load_pca_for_manifold()
feature_blocks = []
meta_rows = []
for row in sorted(main_rows, key=lambda r: int(r['dataset_size'])):
    train_images, train_theta = load_training_with_theta(row)
    heldout_images, heldout_theta = load_heldout_with_theta(row)
    generated_images, generated_theta = load_generated_with_theta(row)
    add_manifold_block(feature_blocks, meta_rows, pca_basis, row=row, source='training real', images=train_images, theta=train_theta, max_n=MANIFOLD_MAX_TRAIN)
    add_manifold_block(feature_blocks, meta_rows, pca_basis, row=row, source='held-out real', images=heldout_images, theta=heldout_theta, max_n=MANIFOLD_MAX_HELDOUT)
    add_manifold_block(feature_blocks, meta_rows, pca_basis, row=row, source='generated', images=generated_images, theta=generated_theta, max_n=MANIFOLD_MAX_GENERATED)

features = np.concatenate(feature_blocks, axis=0)
coords, method_name = reduce_manifold(features)
manifold = pd.DataFrame(meta_rows)
manifold['x'] = coords[:, 0]
manifold['y'] = coords[:, 1]
manifold['method'] = method_name
manifold['pca_rank'] = pca_rank
manifold['pca_explained_variance_sum'] = pca_ev
manifold['pca_basis_sha256'] = pca_sha
manifold_out = CAL_DIR / 'bias_probe_pca_embedding_manifold.csv'
manifold.to_csv(manifold_out, index=False)
print('wrote', manifold_out)
print('manifold method:', method_name, 'n_points:', len(manifold))
display(manifold.groupby(['regime', 'source']).size().rename('n').reset_index())

source_styles = {
    'training real': dict(color='0.55', marker='.', alpha=0.42, s=22),
    'held-out real': dict(color='black', marker='o', alpha=0.72, s=24),
    'generated': dict(color='#d62728', marker='x', alpha=0.82, s=34),
}
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.0), constrained_layout=True)
for ax, regime in zip(axes, ['memorization', 'generalization']):
    sub_regime = manifold[manifold['regime'] == regime]
    if sub_regime.empty:
        ax.set_visible(False)
        continue
    for source, style in source_styles.items():
        sub = sub_regime[sub_regime['source'] == source]
        if sub.empty:
            continue
        ax.scatter(sub['x'], sub['y'], label=source, **style)
    ax.set_title(REGIME_LABELS.get(regime, regime))
    ax.set_xlabel(f'{method_name} 1')
    ax.set_ylabel(f'{method_name} 2')
    ax.grid(alpha=0.12)
    ax.legend(frameon=False, loc='best')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('PCA embedding manifold: source comparison', y=1.04)
source_fig_path = CAL_DIR / 'bias_probe_pca_embedding_manifold_by_source.png'
fig.savefig(source_fig_path, bbox_inches='tight')
print('wrote', source_fig_path)
display(Image(filename=str(source_fig_path)))
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(18.2, 10.4), constrained_layout=True)
real_sources = {'training real', 'held-out real'}
for ax, param in zip(axes.ravel(), PARAM_ORDER):
    real = manifold[manifold['source'].isin(real_sources)]
    gen = manifold[manifold['source'] == 'generated']
    vmin = float(real[param].min())
    vmax = float(real[param].max())
    sc = ax.scatter(real['x'], real['y'], c=real[param], s=18, alpha=0.55, cmap='viridis', vmin=vmin, vmax=vmax, label='real')
    if not gen.empty:
        ax.scatter(gen['x'], gen['y'], c=gen[param], s=25, marker='x', alpha=0.75, cmap='viridis', vmin=vmin, vmax=vmax, label='generated input')
    ax.set_title(PARAM_LABELS.get(param, param))
    ax.set_xlabel(f'{method_name} 1')
    ax.set_ylabel(f'{method_name} 2')
    ax.grid(alpha=0.10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
    cbar.ax.tick_params(labelsize=10)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03))
fig.suptitle('PCA embedding manifold colored by input cosmology', y=1.08)
param_fig_path = CAL_DIR / 'bias_probe_pca_embedding_manifold_parameter_colormaps.png'
fig.savefig(param_fig_path, bbox_inches='tight')
print('wrote', param_fig_path)
display(Image(filename=str(param_fig_path)))
plt.show()

manifold_meta = {
    'method': method_name,
    'pca_basis_path': str(pca_basis_path),
    'pca_rank': int(pca_rank),
    'pca_explained_variance_sum': float(pca_ev),
    'pca_basis_sha256': pca_sha,
    'max_train_per_regime': int(MANIFOLD_MAX_TRAIN),
    'max_heldout_real_per_regime': int(MANIFOLD_MAX_HELDOUT),
    'max_generated_per_regime': int(MANIFOLD_MAX_GENERATED),
}
meta_out = CAL_DIR / 'bias_probe_pca_embedding_manifold_metadata.json'
meta_out.write_text(json.dumps(manifold_meta, indent=2) + '\n')
print('wrote', meta_out)



## Poster-Style Manifold Density View

This is a data-driven version of Nick's schematic. The orange background is the empirical real HI slice density in the PCA+t-SNE feature plane, using training plus held-out real slices. The colored contours and x markers are generated samples. If the small-N model is memorizing/collapsing, generated samples should appear as compact islands rather than filling the real density smoothly.


In [ ]:

if 'manifold' not in globals():
    raise RuntimeError('Run the PCA embedding manifold cell first; it defines manifold.')

DENSITY_BINS = int(os.environ.get('BIAS_PROBE_MANIFOLD_DENSITY_BINS', 160))
DENSITY_SMOOTH_SIGMA = float(os.environ.get('BIAS_PROBE_MANIFOLD_DENSITY_SIGMA', 2.0))


def smooth_hist2d(x, y, *, bins, xrange, yrange):
    h, xe, ye = np.histogram2d(np.asarray(x), np.asarray(y), bins=bins, range=[xrange, yrange], density=True)
    try:
        from scipy.ndimage import gaussian_filter
        h = gaussian_filter(h, sigma=DENSITY_SMOOTH_SIGMA)
    except Exception:
        pass
    xc = 0.5 * (xe[:-1] + xe[1:])
    yc = 0.5 * (ye[:-1] + ye[1:])
    return xc, yc, h.T


def contour_levels(density, qs=(0.65, 0.82, 0.93)):
    vals = np.asarray(density, dtype=float).ravel()
    vals = vals[np.isfinite(vals) & (vals > 0)]
    if len(vals) == 0:
        return []
    levels = np.quantile(vals, qs)
    return sorted(np.unique(levels))

all_x = manifold['x'].to_numpy(float)
all_y = manifold['y'].to_numpy(float)
xpad = 0.06 * max(float(all_x.max() - all_x.min()), 1e-6)
ypad = 0.06 * max(float(all_y.max() - all_y.min()), 1e-6)
xrange = (float(all_x.min() - xpad), float(all_x.max() + xpad))
yrange = (float(all_y.min() - ypad), float(all_y.max() + ypad))

fig, axes = plt.subplots(1, 2, figsize=(16.0, 6.4), constrained_layout=True)
for ax, regime in zip(axes, ['memorization', 'generalization']):
    sub = manifold[manifold['regime'] == regime]
    real = sub[sub['source'].isin(['training real', 'held-out real'])]
    gen = sub[sub['source'] == 'generated']
    if sub.empty or real.empty or gen.empty:
        ax.set_visible(False)
        continue

    xc, yc, real_density = smooth_hist2d(real['x'], real['y'], bins=DENSITY_BINS, xrange=xrange, yrange=yrange)
    _, _, gen_density = smooth_hist2d(gen['x'], gen['y'], bins=DENSITY_BINS, xrange=xrange, yrange=yrange)

    vmax = np.quantile(real_density[real_density > 0], 0.995) if np.any(real_density > 0) else None
    ax.contourf(xc, yc, real_density, levels=28, cmap='Oranges', alpha=0.86, vmax=vmax)
    real_levels = contour_levels(real_density, qs=(0.75, 0.90, 0.975))
    if len(real_levels):
        ax.contour(xc, yc, real_density, levels=real_levels, colors='white', linewidths=1.2, alpha=0.70)

    gen_levels = contour_levels(gen_density, qs=(0.60, 0.78, 0.90, 0.97))
    if len(gen_levels):
        ax.contour(xc, yc, gen_density, levels=gen_levels, colors='#7b2cbf', linewidths=2.0, alpha=0.95)

    ax.scatter(gen['x'], gen['y'], marker='x', s=22, lw=1.4, color='#d62728', alpha=0.58, label='generated samples')
    ax.scatter(real[real['source'] == 'held-out real']['x'], real[real['source'] == 'held-out real']['y'],
               s=12, color='black', alpha=0.38, label='held-out real')
    ax.set_xlim(*xrange)
    ax.set_ylim(*yrange)
    ax.set_xlabel(f'{method_name} 1')
    ax.set_ylabel(f'{method_name} 2')
    ax.set_title(REGIME_LABELS.get(regime, regime))
    ax.text(0.03, 0.04,
            'orange: real p(x) density\npurple: generated density',
            transform=ax.transAxes, ha='left', va='bottom', fontsize=12,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.72, boxstyle='round,pad=0.35'))
    ax.grid(alpha=0.08)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, loc='upper right')

fig.suptitle('PCA+t-SNE manifold density: real distribution vs generated samples', y=1.04)
density_path = CAL_DIR / 'bias_probe_pca_embedding_manifold_density_schematic.png'
fig.savefig(density_path, bbox_inches='tight')
print('wrote', density_path)
display(Image(filename=str(density_path)))
plt.show()



## Direct PCA Mode Diagnostics

The manifold plot is nonlinear, so it can sometimes exaggerate or hide structure. This section looks directly at the frozen PCA basis used by the encoder.

What this checks:

- The mode spectrum shows how many PCA directions are needed for the 98% basis.
- The PC1/PC2 plane shows whether the largest-variance PCA directions already separate cosmology values.
- The correlation heatmap asks a sharper question: which PCA coefficients are correlated with each CAMELS parameter on real data only? Smooth/correlated structure means the PCA encoder has cosmology information available. If the generated points do not follow that structure, the issue is more likely the diffusion model / conditioning than the PCA basis itself.


In [ ]:

PCA_MODE_CORR_N = int(os.environ.get('BIAS_PROBE_PCA_MODE_CORR_N', 32))

if 'features' not in globals() or 'manifold' not in globals() or 'pca_basis' not in globals():
    raise RuntimeError('Run the PCA embedding manifold cell first; it defines features, manifold, and pca_basis.')

# 1. PCA mode spectrum for the exact frozen basis used by the encoder.
evr = np.asarray(pca_basis.explained_variance_ratio, dtype=float)
components = np.arange(1, len(evr) + 1)
cumulative = np.cumsum(evr)
strength = np.sqrt(np.maximum(evr, 0.0))

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.4), constrained_layout=True)
axes[0].plot(components, strength, lw=2.4, color='#3b4cc0')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('PCA mode index')
axes[0].set_ylabel(r'Mode strength $\sqrt{\lambda_i / \sum_j \lambda_j}$')
axes[0].set_title('PCA mode strength')
axes[0].grid(alpha=0.15)
axes[1].plot(components, cumulative, lw=2.8, color='#b40426')
axes[1].axhline(0.98, color='black', ls=':', lw=1.8, label='98% target')
axes[1].set_xscale('log')
axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel('PCA modes retained')
axes[1].set_ylabel('Cumulative explained variance')
axes[1].set_title('Variance captured')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.15)
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle(f'Frozen PCA basis: {len(evr):,} modes, explained variance={float(cumulative[-1]):.4f}')
pca_spectrum_path = CAL_DIR / 'bias_probe_pca_mode_spectrum.png'
fig.savefig(pca_spectrum_path, bbox_inches='tight')
print('wrote', pca_spectrum_path)
display(Image(filename=str(pca_spectrum_path)))
plt.show()

# 2. Direct PC1/PC2 plane colored by cosmology. This is not t-SNE/UMAP: axes are actual PCA coefficients.
mode_df = manifold.copy()
mode_df['PC1'] = features[:, 0]
mode_df['PC2'] = features[:, 1]
mode_df['PC3'] = features[:, 2] if features.shape[1] > 2 else np.nan
real_sources = {'training real', 'held-out real'}
real = mode_df[mode_df['source'].isin(real_sources)]
gen = mode_df[mode_df['source'] == 'generated']

fig, axes = plt.subplots(2, 3, figsize=(18.2, 10.4), constrained_layout=True)
for ax, param in zip(axes.ravel(), PARAM_ORDER):
    vmin = float(real[param].min())
    vmax = float(real[param].max())
    sc = ax.scatter(real['PC1'], real['PC2'], c=real[param], s=18, alpha=0.55, cmap='viridis', vmin=vmin, vmax=vmax, label='real')
    if not gen.empty:
        ax.scatter(gen['PC1'], gen['PC2'], c=gen[param], s=28, marker='x', alpha=0.75, cmap='viridis', vmin=vmin, vmax=vmax, label='generated input')
    ax.set_title(PARAM_LABELS.get(param, param))
    ax.set_xlabel('PC1 coefficient')
    ax.set_ylabel('PC2 coefficient')
    ax.grid(alpha=0.10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
    cbar.ax.tick_params(labelsize=10)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03))
fig.suptitle('Direct PCA mode plane colored by input cosmology', y=1.08)
pca_plane_path = CAL_DIR / 'bias_probe_pca_mode_plane_parameter_colormaps.png'
fig.savefig(pca_plane_path, bbox_inches='tight')
print('wrote', pca_plane_path)
display(Image(filename=str(pca_plane_path)))
plt.show()

# 3. Correlate individual PCA coefficients with cosmology values on real data only.
# Use Spearman correlation because the parameter relation can be monotonic but not linear.
real_mask = mode_df['source'].isin(real_sources).to_numpy()
top_n = min(PCA_MODE_CORR_N, features.shape[1])
corr = np.empty((top_n, len(PARAM_ORDER)), dtype=float)
for i in range(top_n):
    pc = pd.Series(features[real_mask, i])
    for j, param in enumerate(PARAM_ORDER):
        corr[i, j] = pc.corr(pd.Series(mode_df.loc[real_mask, param].to_numpy(float)), method='spearman')

corr_df = pd.DataFrame(corr, columns=PARAM_ORDER)
corr_df.insert(0, 'mode', np.arange(1, top_n + 1))
corr_out = CAL_DIR / 'bias_probe_pca_mode_cosmology_correlations.csv'
corr_df.to_csv(corr_out, index=False)
print('wrote', corr_out)
display(corr_df.head(12).round(3))

fig, ax = plt.subplots(figsize=(10.8, max(6.0, 0.28 * top_n)), constrained_layout=True)
im = ax.imshow(corr, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(PARAM_ORDER)), [PARAM_LABELS.get(p, p) for p in PARAM_ORDER])
ax.set_yticks(np.arange(top_n), [f'PC{i}' for i in range(1, top_n + 1)])
ax.set_xlabel('Cosmology parameter')
ax.set_ylabel('PCA coefficient')
ax.set_title('Spearman correlation: PCA modes vs real cosmology labels')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label('Spearman r')
pca_corr_path = CAL_DIR / 'bias_probe_pca_mode_cosmology_correlations.png'
fig.savefig(pca_corr_path, bbox_inches='tight')
print('wrote', pca_corr_path)
display(Image(filename=str(pca_corr_path)))
plt.show()

# A compact ranking: which parameters are most visible in the top PCA coefficients?
rank_rows = []
for j, param in enumerate(PARAM_ORDER):
    abs_corr = np.abs(corr[:, j])
    best_i = int(np.nanargmax(abs_corr))
    rank_rows.append({
        'parameter': param,
        'label': PARAM_LABELS.get(param, param),
        'best_mode': best_i + 1,
        'best_spearman_r': float(corr[best_i, j]),
        'best_abs_spearman_r': float(abs_corr[best_i]),
        'mean_abs_top_modes': float(np.nanmean(abs_corr)),
    })
pca_corr_summary = pd.DataFrame(rank_rows).sort_values('best_abs_spearman_r', ascending=False)
pca_corr_summary_out = CAL_DIR / 'bias_probe_pca_mode_cosmology_correlation_summary.csv'
pca_corr_summary.to_csv(pca_corr_summary_out, index=False)
print('wrote', pca_corr_summary_out)
display(pca_corr_summary.round(3))


## Optional: Compare SSCD Or Another Encoder

The main calibration plot above now uses the best VGG probe. This optional section is only for checking whether conclusions change under another image embedding.

The older SSCD+Ridge encoder uses the same real-only split and the same held-out cosmologies; only the image embedding changes.

Great Lakes commands:

```bash
cd /home/jiamingp/diffusion_models_repo
sscd_enc=$(sbatch -A huterer2 --parsable scripts/slurm/train_nf_conditional_bias_sscd_encoder.sbatch)
sscd_eval=$(sbatch -A huterer2 --parsable --dependency=afterok:${sscd_enc} scripts/slurm/evaluate_nf_conditional_bias_sscd_probe.sbatch)
echo "sscd_enc=$sscd_enc sscd_eval=$sscd_eval"
```

After those finish, this cell will automatically load `results/nf_conditional_bias_probe/sscd_calibration`. You can also point to another encoder output by setting `BIAS_PROBE_ALT_CAL_DIR`.

In [ ]:

alt_dir_env = os.environ.get('BIAS_PROBE_ALT_CAL_DIR', '').strip()
if alt_dir_env:
    candidate_dirs = [Path(alt_dir_env).expanduser().resolve()]
else:
    candidate_dirs = [RESULT_DIR / 'sscd_calibration']

loaded_any = False
for ALT_CAL_DIR in candidate_dirs:
    alt_points_path = ALT_CAL_DIR / 'bias_probe_per_cosmology_points.csv'
    alt_slopes_path = ALT_CAL_DIR / 'bias_probe_regime_slopes.csv'
    if alt_points_path.exists() and alt_slopes_path.exists():
        alt_points = pd.read_csv(alt_points_path)
        alt_slopes = pd.read_csv(alt_slopes_path)
        print('Loaded alternative encoder calibration from', ALT_CAL_DIR)
        display(with_parameter_labels(alt_slopes.sort_values(['parameter', 'dataset_size'])).round(4))
        plot_clean_calibration(alt_points, alt_slopes, ALT_CAL_DIR / 'bias_probe_calibration_recovered_vs_input_clean.png')
        loaded_any = True
    else:
        print('Alternative calibration not available yet:', ALT_CAL_DIR)

sscd_metrics_path = ENC_DIR / 'sscd_encoder_val_metrics.csv'
if sscd_metrics_path.exists():
    sscd_metrics = pd.read_csv(sscd_metrics_path)
    print('SSCD+Ridge real-validation metrics')
    display(sscd_metrics[sscd_metrics['split'] == 'val'].round(4))

if not loaded_any:
    print('No alternative encoder calibration loaded yet. Run the SSCD sbatch commands above, or set BIAS_PROBE_ALT_CAL_DIR to another calibration output directory.')


## Interpretation Checklist

Use this order when reading the plot:

1. Check the real-data VGG encoder first. If the VGG probe cannot recover a parameter on held-out real simulations, generated-field calibration for that parameter is weak evidence.
2. Lead with `Omega_m`; this is the clearest current result. `sigma_8` is weaker, and the feedback parameters are much less reliable from HI alone.
3. Compare fitted slope to the ideal value `1`. A flat slope means the generated fields are not strongly tracking the requested input parameter.
4. Use the vertical bars as generated-sample diversity at fixed input, not as accuracy. Small bars plus biased medians is still bad calibration.
5. Compare memorization vs generalization. The expected result is that the generalization-regime model has a slope closer to `1` for well-constrained parameters.

Poster sentence:

> Using a frozen VGG16 cosmology probe, the large-data conditional diffusion model recovers the requested `Omega_m` much better than the small-data model, suggesting that the generalization-regime model is more faithful to the conditioning cosmology.